# Two-Regression Pipeline Forecasting Experiment

This notebook implements a simplified forecasting model using **2 XGBoost regressors** instead of the baseline's 4 regressors.

**Regression Targets:**
- `phase3_worse` = phase3% + phase4% + phase5% (probability of phase 3 or worse)
- `phase4_worse` = phase4% + phase5% (probability of phase 4 or worse)

**3-Class Phase Classification (top-down, threshold = 0.2):**

| Condition | Assigned Class | Represents |
|---|---|---|
| phase4_worse >= 0.2 | 4 | Phase 4-5 |
| phase3_worse >= 0.2 | 3 | Phase 3 |
| otherwise | 2 | Phase 1-2 |

This merges phases 1 & 2 into a single class and phases 4 & 5 into a single class,
reducing the classification from 5 to 3 categories.

In [1]:
import numpy as np
import datetime
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from sklearn.metrics import mean_squared_error, accuracy_score, f1_score
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import shap
from xgboost import XGBClassifier
# import random forest regressor
from sklearn.ensemble import RandomForestRegressor
#import linear regression
from sklearn.linear_model import LinearRegression
# import tqdm
from tqdm import tqdm
import tqdm
#import r2_score
from sklearn.metrics import r2_score
#import confusion matrix
from sklearn.metrics import confusion_matrix
# import roc auc score
from sklearn.metrics import roc_auc_score
from ipcch.food_crisis_functions import *
import json
with open("forecasting_hyperparameters.json", "r") as file:
    best_params_xgb_regressor= json.load(file)
    
with open("forecasting_hyperparameters_p3.json", "r") as file:
    best_params_xgb_regressor_for_p3= json.load(file)

def convert_prob_to_phase_two_reg(y_pred_test, th=0.2):
    """
    Convert 2-regression predictions to 3-class phase classification.
    Targets: phase3_worse (phase 3+4+5%) and phase4_worse (phase 4+5%)
    
    Classification (top-down):
    - phase4_worse >= th -> class 4 (represents phase 4-5)
    - phase3_worse >= th -> class 3
    - else -> class 2 (represents phase 1-2)
    """
    y_pred_test['y_pred'] = y_pred_test['y_pred'].round(2)
    
    phase3_data = y_pred_test[y_pred_test['phase'] == 3].reset_index(drop=True)
    phase4_data = y_pred_test[y_pred_test['phase'] == 4].reset_index(drop=True)
    
    result = pd.DataFrame()
    result['phase3_pred'] = phase3_data['y_pred'].values
    result['phase3_test'] = phase3_data['y_test'].values
    result['phase4_pred'] = phase4_data['y_pred'].values
    result['phase4_test'] = phase4_data['y_test'].values
    
    if 'test_index' in phase4_data.columns:
        result['test_index'] = phase4_data['test_index'].values
    
    result = result.fillna(0)
    
    for row, idx in zip(result.itertuples(), result.index):
        # Ground truth
        if row.phase4_test >= th:
            result.loc[idx, 'overall_phase'] = 4
        elif row.phase3_test >= th:
            result.loc[idx, 'overall_phase'] = 3
        else:
            result.loc[idx, 'overall_phase'] = 2
        
        # Prediction
        if row.phase4_pred >= th:
            result.loc[idx, 'overall_phase_pred'] = 4
        elif row.phase3_pred >= th:
            result.loc[idx, 'overall_phase_pred'] = 3
        else:
            result.loc[idx, 'overall_phase_pred'] = 2
    
    return result

def all_metrics_three_class(y_test, y_pred, cm, y_pred_test):
    """
    Metrics for 3-class classification (labels: 2=phase1-2, 3=phase3, 4=phase4-5).
    cm is 3x3 with rows/cols for labels [2, 3, 4].
    
    Sensitivity: recall for phase 3+ (actual 3 or 4-5 correctly classified as 3+)
    Precision: precision for phase 3+ predictions
    R-squared: on phase3_worse continuous predictions
    """
    accuracy_val = accuracy_score(y_test, y_pred)
    
    # Phase 3+ = indices 1,2 in the 3x3 matrix (labels 3 and 4)
    correct_3_more = np.sum(cm[1:, 1:])
    total_3_more = np.sum(cm[1:, :])
    sensitivity = correct_3_more / total_3_more if total_3_more != 0 else 0
    
    precise_3_more = np.sum(cm[1:, 1:])
    total_prec_3_more = np.sum(cm[:, 1:])
    precision = precise_3_more / total_prec_3_more if total_prec_3_more != 0 else 0
    
    overall_r2 = r2_score(y_pred_test['phase3_test'], y_pred_test['phase3_pred'])
    
    return accuracy_val, sensitivity, precision, overall_r2

In [2]:

# read csv
df = pd.read_csv(r'C:\Users\swl00\IFPRI Dropbox\Weilun Shi\Google fund\Analysis\1.Source Data\forecasting_subset_IPCCH_v1210.csv')

# prepare date from year and month
df['date'] = pd.to_datetime(df[['year', 'month']].assign(DAY=1))
# check for inf and replace with na
df = df.replace([np.inf, -np.inf], np.nan)

# replace df['overall_phase], 0 as 1, >5 as 5
df['overall_phase'] = df['overall_phase'].apply(lambda x: 1 if x < 1 else (5 if x > 5 else x))

df_origin = df.copy()
y_pred_test = pd.DataFrame()
model_stats = pd.DataFrame()
#select phase1_percent is not na
df = df_origin[df_origin['phase1_percent'].notna()]

# Sort by region and date
df = df.sort_values(by=['area_id', 'date'])
#drop overall phase
df = df.drop(['overall_phase'], axis=1)

# Two-regression targets only: phase3_worse and phase4_worse
# (baseline uses 4 targets: phase2_worse, phase3_worse, phase4_worse, phase5_worse)
df['phase3_worse'] = df['phase3_percent'] + df['phase4_percent'] + df['phase5_percent']
df['phase4_worse'] = df['phase4_percent'] + df['phase5_percent']
#drop phase2_percent, phase3_percent, phase4_percent, phase5_percent, phase1_percent
df = df.drop(['phase2_percent', 'phase3_percent', 'phase4_percent', 'phase5_percent', 'phase1_percent'], axis=1)


In [3]:
df.to_csv(r'C:\Users\swl00\IFPRI Dropbox\Weilun Shi\Google fund\Analysis\1.Source Data\forecasting_subset_IPCCH_v1210_processed_two_reg.csv', index=False)

## Test Year 2024 (train on 2021-2023, test on 2024)

In [4]:


y_pred_test = pd.DataFrame()
shape_values_df_ensemble = pd.DataFrame()
df_result = pd.DataFrame()
date = "2024-01-01"  # Define the 'date' variable
y_pred_test=pd.DataFrame()

# Two-regression pipeline: only phase 3 and phase 4 targets
for i in [3, 4]:
    train_df = df[df['date'] < date]
    test_df = df[df['date'] >= date]
    train_df = train_df.drop(['date','area_id'], axis=1)
    test_df = test_df.drop(['date','area_id'], axis=1)
    train_df_new = train_df.drop(['phase{}_worse'.format(j) for j in [3, 4] if j != i], axis=1)
    test_df_new = test_df.drop(['phase{}_worse'.format(j) for j in [3, 4] if j != i], axis=1)
# drop rows with NaN in phase{}_percent
    train_df_new = train_df_new.dropna(subset=['phase{}_worse'.format(i)])
    test_df_new = test_df_new.dropna(subset=['phase{}_worse'.format(i)])
    test_index = test_df_new.index
    # Split into features and target
    X_train = train_df_new.drop('phase{}_worse'.format(i), axis=1)
    y_train = train_df_new['phase{}_worse'.format(i)]
    X_test = test_df_new.drop('phase{}_worse'.format(i), axis=1)
    y_test = test_df_new['phase{}_worse'.format(i)]
    if i == 3:
        best_params_xgb_regressor = best_params_xgb_regressor_for_p3
    model = xgb.XGBRegressor(**best_params_xgb_regressor)
    model.fit(X_train, y_train)
    # Predictions
    y_pred = model.predict(X_test)
    # for y_pred_test, add a column to indicate the phase
    # test_index is recorded on the last iteration (phase 4) for later merge
    if i != 4:
        y_pred_test = pd.concat([y_pred_test, pd.DataFrame({'y_pred': y_pred, 'y_test': y_test, 'phase': [i]*len(y_pred)})], ignore_index=True)
    else:
        y_pred_test = pd.concat([y_pred_test, pd.DataFrame({'y_pred': y_pred, 'y_test': y_test, 'phase': [i]*len(y_pred),'test_index':test_index})], ignore_index=True)
   
    # shap values
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_train)
    
    #save shap values
    shap_values_df = pd.DataFrame(shap_values, columns=X_train.columns)
    
    # add a column to indicate the phase
    shap_values_df['phase'] = i
    
    # append to shape_values_df_ensemble
    shape_values_df_ensemble = pd.concat([shape_values_df_ensemble, shap_values_df], ignore_index=True)

# Convert to 3-class phase classification (2=phase1-2, 3=phase3, 4=phase4-5)
y_pred_test = convert_prob_to_phase_two_reg(y_pred_test)
y_test = y_pred_test['overall_phase']
y_pred = y_pred_test['overall_phase_pred']

# confusion matrix (3-class with explicit labels)
cm = confusion_matrix(y_test, y_pred, labels=[2, 3, 4])

accuracy_score_new, sensitivity, precision, overall_r2 = all_metrics_three_class(y_test, y_pred, cm, y_pred_test)

print("Accuracy:", accuracy_score_new)
print("Sensitivity:", sensitivity)
print("Precision:", precision)
print("Overall R\u00b2(Phase 3 or more):", overall_r2)

Accuracy: 0.7602230483271375
Sensitivity: 0.829763866007688
Precision: 0.7250479846449136
Overall R²(Phase 3 or more): 0.472545928594285


Save y_pred_test to csv (optional)

In [ ]:
# remove column if it is all 0
y_pred_test = y_pred_test.loc[:, (y_pred_test != 0).any(axis=0)]
#extract df area_id, date
df_extracted = df[['area_id', 'date']]

# generate index
df_extracted['test_index'] = df_extracted.index

# merge y_pred_test with df_extracted on test_index
y_pred_test = y_pred_test.merge(df_extracted, on='test_index', how='left')
#read IPCCH
PATH = r'C:\Users\swl00\IFPRI Dropbox\Weilun Shi\Google fund\Analysis\1.Source Data\IPCCH_2017_2025_final_v12102025_with_zscores.csv'
df_ipcch = pd.read_csv(PATH)
# keep only admin_code, lat, lon
df_ipcch = df_ipcch[['admin_code', 'lat', 'lon']]

# rename admin_code to area_id
df_ipcch = df_ipcch.rename(columns={'admin_code': 'area_id'})

# merge y_pred_test with df_ipcch on area_id
y_pred_test = y_pred_test.merge(df_ipcch, on='area_id', how='left')
# save y_pred_test to csv
y_pred_test.to_csv(r'results/predictions/forecasting/forecasting_two_reg_y_pred_test_2024.csv', index=False)
# clean all memory
%reset -f

## Test Year 2023 (train on 2020-2022, test on 2023)

In [5]:
# read csv
df = pd.read_csv(r'C:\Users\swl00\IFPRI Dropbox\Weilun Shi\Google fund\Analysis\1.Source Data\forecasting_subset_IPCCH_v1210.csv')

# prepare date from year and month
df['date'] = pd.to_datetime(df[['year', 'month']].assign(DAY=1))
# check for inf and replace with na
df = df.replace([np.inf, -np.inf], np.nan)

# replace df['overall_phase], 0 as 1, >5 as 5
df['overall_phase'] = df['overall_phase'].apply(lambda x: 1 if x < 1 else (5 if x > 5 else x))

df_origin = df.copy()
y_pred_test = pd.DataFrame()
model_stats = pd.DataFrame()
#select phase1_percent is not na
df = df_origin[df_origin['phase1_percent'].notna()]

# Sort by region and date
df = df.sort_values(by=['area_id', 'date'])
#drop overall phase
df = df.drop(['overall_phase'], axis=1)

# Two-regression targets only
df['phase3_worse'] = df['phase3_percent'] + df['phase4_percent'] + df['phase5_percent']
df['phase4_worse'] = df['phase4_percent'] + df['phase5_percent']
df = df.drop(['phase2_percent', 'phase3_percent', 'phase4_percent', 'phase5_percent', 'phase1_percent'], axis=1)

y_pred_test = pd.DataFrame()
shape_values_df_ensemble = pd.DataFrame()
df_result = pd.DataFrame()
date = "2023-01-01"  # Define the 'date' variable
cutoff_date = "2024-01-01"
y_pred_test=pd.DataFrame()

for i in [3, 4]:
    train_df = df[df['date'] < date]
    test_df = df[(df['date'] < cutoff_date) & (df['date'] >= date)]
    train_df = train_df.drop(['date','area_id'], axis=1)
    test_df = test_df.drop(['date','area_id'], axis=1)
    train_df_new = train_df.drop(['phase{}_worse'.format(j) for j in [3, 4] if j != i], axis=1)
    test_df_new = test_df.drop(['phase{}_worse'.format(j) for j in [3, 4] if j != i], axis=1)
# drop rows with NaN in phase{}_percent
    train_df_new = train_df_new.dropna(subset=['phase{}_worse'.format(i)])
    test_df_new = test_df_new.dropna(subset=['phase{}_worse'.format(i)])
    test_index = test_df_new.index
    # Split into features and target
    X_train = train_df_new.drop('phase{}_worse'.format(i), axis=1)
    y_train = train_df_new['phase{}_worse'.format(i)]
    X_test = test_df_new.drop('phase{}_worse'.format(i), axis=1)
    y_test = test_df_new['phase{}_worse'.format(i)]
    if i == 3:
        best_params_xgb_regressor = best_params_xgb_regressor_for_p3
    model = xgb.XGBRegressor(**best_params_xgb_regressor)
    model.fit(X_train, y_train)
    # Predictions
    y_pred = model.predict(X_test)
    if i != 4:
        y_pred_test = pd.concat([y_pred_test, pd.DataFrame({'y_pred': y_pred, 'y_test': y_test, 'phase': [i]*len(y_pred)})], ignore_index=True)
    else:
        y_pred_test = pd.concat([y_pred_test, pd.DataFrame({'y_pred': y_pred, 'y_test': y_test, 'phase': [i]*len(y_pred),'test_index':test_index})], ignore_index=True)
    # shap values
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_train)
    shap_values_df = pd.DataFrame(shap_values, columns=X_train.columns)
    shap_values_df['phase'] = i
    shape_values_df_ensemble = pd.concat([shape_values_df_ensemble, shap_values_df], ignore_index=True)

y_pred_test = convert_prob_to_phase_two_reg(y_pred_test)
y_test = y_pred_test['overall_phase']
y_pred = y_pred_test['overall_phase_pred']

# confusion matrix
cm = confusion_matrix(y_test, y_pred, labels=[2, 3, 4])

accuracy_score_new, sensitivity, precision, overall_r2 = all_metrics_three_class(y_test, y_pred, cm, y_pred_test)

print("Accuracy:", accuracy_score_new)
print("Sensitivity:", sensitivity)
print("Precision:", precision)
print("Overall R\u00b2(Phase 3 or more):", overall_r2)

Accuracy: 0.7710696920583469
Sensitivity: 0.815444390480816
Precision: 0.7542677448337826
Overall R²(Phase 3 or more): 0.5373951390663174


In [ ]:
# remove column if it is all 0
y_pred_test = y_pred_test.loc[:, (y_pred_test != 0).any(axis=0)]
#extract df area_id, date
df_extracted = df[['area_id', 'date']]

# generate index
df_extracted['test_index'] = df_extracted.index

# merge y_pred_test with df_extracted on test_index
y_pred_test = y_pred_test.merge(df_extracted, on='test_index', how='left')
#read IPCCH
PATH = r'C:\Users\swl00\IFPRI Dropbox\Weilun Shi\Google fund\Analysis\1.Source Data\IPCCH_2017_2025_final_v12102025_with_zscores.csv'
df_ipcch = pd.read_csv(PATH)
# keep only admin_code, lat, lon
df_ipcch = df_ipcch[['admin_code', 'lat', 'lon']]

# rename admin_code to area_id
df_ipcch = df_ipcch.rename(columns={'admin_code': 'area_id'})

# merge y_pred_test with df_ipcch on area_id
y_pred_test = y_pred_test.merge(df_ipcch, on='area_id', how='left')
# save y_pred_test to csv
y_pred_test.to_csv(r'results/predictions/forecasting/forecasting_two_reg_y_pred_test_2023.csv', index=False)
# clean all memory
%reset -f

## Test Year 2022 (train on 2019-2021, test on 2022)

In [6]:
# read csv
df = pd.read_csv(r'C:\Users\swl00\IFPRI Dropbox\Weilun Shi\Google fund\Analysis\1.Source Data\forecasting_subset_IPCCH_v1210.csv')

# prepare date from year and month
df['date'] = pd.to_datetime(df[['year', 'month']].assign(DAY=1))
# check for inf and replace with na
df = df.replace([np.inf, -np.inf], np.nan)

# replace df['overall_phase], 0 as 1, >5 as 5
df['overall_phase'] = df['overall_phase'].apply(lambda x: 1 if x < 1 else (5 if x > 5 else x))

df_origin = df.copy()
y_pred_test = pd.DataFrame()
model_stats = pd.DataFrame()
#select phase1_percent is not na
df = df_origin[df_origin['phase1_percent'].notna()]

# Sort by region and date
df = df.sort_values(by=['area_id', 'date'])
#drop overall phase
df = df.drop(['overall_phase'], axis=1)

# Two-regression targets only
df['phase3_worse'] = df['phase3_percent'] + df['phase4_percent'] + df['phase5_percent']
df['phase4_worse'] = df['phase4_percent'] + df['phase5_percent']
df = df.drop(['phase2_percent', 'phase3_percent', 'phase4_percent', 'phase5_percent', 'phase1_percent'], axis=1)

y_pred_test = pd.DataFrame()
shape_values_df_ensemble = pd.DataFrame()
df_result = pd.DataFrame()
date = "2022-01-01"  # Define the 'date' variable
cutoff_date = "2023-01-01"
y_pred_test=pd.DataFrame()

for i in [3, 4]:
    train_df = df[df['date'] < date]
    test_df = df[(df['date'] < cutoff_date) & (df['date'] >= date)]
    train_df = train_df.drop(['date','area_id'], axis=1)
    test_df = test_df.drop(['date','area_id'], axis=1)
    train_df_new = train_df.drop(['phase{}_worse'.format(j) for j in [3, 4] if j != i], axis=1)
    test_df_new = test_df.drop(['phase{}_worse'.format(j) for j in [3, 4] if j != i], axis=1)
# drop rows with NaN in phase{}_percent
    train_df_new = train_df_new.dropna(subset=['phase{}_worse'.format(i)])
    test_df_new = test_df_new.dropna(subset=['phase{}_worse'.format(i)])
    test_index = test_df_new.index
    # Split into features and target
    X_train = train_df_new.drop('phase{}_worse'.format(i), axis=1)
    y_train = train_df_new['phase{}_worse'.format(i)]
    X_test = test_df_new.drop('phase{}_worse'.format(i), axis=1)
    y_test = test_df_new['phase{}_worse'.format(i)]
    if i == 3:
        best_params_xgb_regressor = best_params_xgb_regressor_for_p3
    model = xgb.XGBRegressor(**best_params_xgb_regressor)
    model.fit(X_train, y_train)
    # Predictions
    y_pred = model.predict(X_test)
    if i != 4:
        y_pred_test = pd.concat([y_pred_test, pd.DataFrame({'y_pred': y_pred, 'y_test': y_test, 'phase': [i]*len(y_pred)})], ignore_index=True)
    else:
        y_pred_test = pd.concat([y_pred_test, pd.DataFrame({'y_pred': y_pred, 'y_test': y_test, 'phase': [i]*len(y_pred),'test_index':test_index})], ignore_index=True)
    # shap values
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_train)
    shap_values_df = pd.DataFrame(shap_values, columns=X_train.columns)
    shap_values_df['phase'] = i
    shape_values_df_ensemble = pd.concat([shape_values_df_ensemble, shap_values_df], ignore_index=True)

y_pred_test = convert_prob_to_phase_two_reg(y_pred_test)
y_test = y_pred_test['overall_phase']
y_pred = y_pred_test['overall_phase_pred']

# confusion matrix
cm = confusion_matrix(y_test, y_pred, labels=[2, 3, 4])

accuracy_score_new, sensitivity, precision, overall_r2 = all_metrics_three_class(y_test, y_pred, cm, y_pred_test)

print("Accuracy:", accuracy_score_new)
print("Sensitivity:", sensitivity)
print("Precision:", precision)
print("Overall R\u00b2(Phase 3 or more):", overall_r2)

Accuracy: 0.6698292220113852
Sensitivity: 0.5924838520258368
Precision: 0.7156028368794326
Overall R²(Phase 3 or more): 0.31223746076386494


In [ ]:
# remove column if it is all 0
y_pred_test = y_pred_test.loc[:, (y_pred_test != 0).any(axis=0)]
#extract df area_id, date
df_extracted = df[['area_id', 'date']]

# generate index
df_extracted['test_index'] = df_extracted.index

# merge y_pred_test with df_extracted on test_index
y_pred_test = y_pred_test.merge(df_extracted, on='test_index', how='left')
#read IPCCH
PATH = r'C:\Users\swl00\IFPRI Dropbox\Weilun Shi\Google fund\Analysis\1.Source Data\IPCCH_2017_2025_final_v12102025_with_zscores.csv'
df_ipcch = pd.read_csv(PATH)
# keep only admin_code, lat, lon
df_ipcch = df_ipcch[['admin_code', 'lat', 'lon']]

# rename admin_code to area_id
df_ipcch = df_ipcch.rename(columns={'admin_code': 'area_id'})

# merge y_pred_test with df_ipcch on area_id
y_pred_test = y_pred_test.merge(df_ipcch, on='area_id', how='left')
# save y_pred_test to csv
y_pred_test.to_csv(r'results/predictions/forecasting/forecasting_two_reg_y_pred_test_2022.csv', index=False)
# clean all memory
%reset -f

## Visualization (uses data from the last test year run above)

In [ ]:
#calculate F1 score
f1 = f1_score(y_test, y_pred, average='weighted')
print("F1 Score:", f1)

In [ ]:
# see df_origin overall phase distribution (original 5-class)
df_origin['overall_phase'].value_counts()

In [ ]:
# plot df_origin each phase distribution with percentage of total label on top
phase_counts = df_origin['overall_phase'].value_counts().sort_index()
plt.figure(figsize=(8, 6))
sns.barplot(x=phase_counts.index, y=phase_counts.values, palette="viridis")
plt.title('Distribution of Overall Phase (Original 5-Class)')
plt.xlabel('Overall Phase')
plt.ylabel('Count')
# add percentage on top of bars
total = sum(phase_counts.values)
for i, v in enumerate(phase_counts.values):
    plt.text(i, v + 0.5, f"{v} ({v/total:.1%})", ha='center')
plt.ylim(0, max(phase_counts.values) * 1.1)
plt.show()

In [ ]:
# see 3-class confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['1-2', '3', '4-5'], yticklabels=['1-2', '3', '4-5'])
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix (3-Class: Phase 1-2, 3, 4-5)')
plt.show()

In [ ]:
# plot AUC ROC curve
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc
# define >=3 as positive class (phase 3+ vs phase 1-2)
y_test_binary = (y_test >= 3).astype(int)
y_pred_binary = (y_pred >= 3).astype(int)
fpr, tpr, thresholds = roc_curve(y_test_binary, y_pred_binary)
roc_auc = auc(fpr, tpr)
plt.figure()
plt.plot(fpr, tpr, color='darkorange', lw=2, label='ROC curve (area = %0.2f)' % roc_auc)
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (Phase 3+ vs 1-2)')
plt.legend(loc="lower right")
plt.show()

In [ ]:
# precision - recall curve by bootstrapping
from sklearn.metrics import precision_recall_curve, average_precision_score
from sklearn.utils import resample

n_iterations = 1000
bootstrapped_scores = []
np.random.seed(42)
precision_list = []
recall_list = []
for i in range(n_iterations):
    # Bootstrap sample
    y_test_bs, y_pred_bs = resample(y_test, y_pred, random_state=i)
    y_test_binary = (y_test_bs >= 3).astype(int)
    y_pred_binary = (y_pred_bs >= 3).astype(int)
    precision, recall, _ = precision_recall_curve(y_test_binary, y_pred_binary)
    precision_list.append(precision)
    recall_list.append(recall)
    bootstrapped_scores.append(average_precision_score(y_test_binary, y_pred_binary))

# plot precision - recall curve
plt.figure()
for i in range(n_iterations):
    plt.plot(recall_list[i], precision_list[i], color='lightgray', alpha=0.3)
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve (Phase 3+ vs 1-2)')
plt.show()
